## tl;dr

- 2026-07-24 至 2026-07-30，四条活动合计花费 **$2,457.19**、归因销售 **$8,115.00**、购买 **270**，整体 ROAS **3.30**。
- 四条活动的每日花费都高度贴近固定水平，且 KD5、Y180、S75-wide 多数日期在下午提前停止获得流量，构成很强的预算截断信号；但导出不含预算金额或预算状态，因此仍是推断。
- 最值得直接做小幅加价测试的窗口是：KD5 09–11/13 点、Y180 04/06–07 点、S75-wide 08–09/11–12/15 点、S75-sneakers 16/18–19 点。
- 最值得先延展覆盖、暂不加价的窗口是：KD5 18–20 点、Y180 18–20 点、S75-wide 18 点、S75-sneakers 21 点。
- 搜索词表的点击、花费、购买、销售与 TIB 按活动精确一致；`日期范围`是每个唯一搜索词的观察跨度，不能只取完整 7 天行。

## Context & Methods

本分析服务于分时竞价决策，区分三种动作：继续覆盖、提高竞价、保持或压低。小时表现统一用汇总销售额 ÷ 汇总花费计算 ROAS，不平均行级 ROAS；稳定性同时参考点击量、购买量、出单日期数和获得展示的日期数。

### Key Assumptions

- 小时按 Amazon 导出中的账户报告时区解释；文件未提供具体时区。
- 由于文件没有预算金额、预算耗尽状态和变更日志，“预算截断”只能根据每日花费平台化与最后活跃小时推断。
- 没有盈亏平衡 ROAS，因此“适合加价”表示相对该活动基线更高且样本较稳定，不等同于已确认盈利。

## Data

### 1. Load and normalize the two read-only CSV exports

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
def display(value):
    print(value)

TIB_PATH = Path('/Users/panjinlong/Library/Application Support/ziniaobrowserdatas/ziniao browser/欧德思美站/TIB优化 (1).csv')
SEARCH_PATH = Path('/Users/panjinlong/Library/Application Support/ziniaobrowserdatas/ziniao browser/欧德思美站/4广告搜索词.csv')
MEASURES = ['展示量', '点击量', '总成本', '购买量', '销售额']

tib = pd.read_csv(TIB_PATH, encoding='utf-8-sig')
search = pd.read_csv(SEARCH_PATH, encoding='utf-8-sig')
source_shapes = {'TIB': tib.shape, '搜索词': search.shape}

for frame in (tib, search):
    for col in MEASURES:
        frame[col] = pd.to_numeric(frame[col], errors='coerce').fillna(0.0)

tib['date'] = pd.to_datetime(tib['日期'], format='%Y年%m月%d日')
tib['hour'] = pd.to_numeric(tib['小时'], errors='coerce').astype(int)
tib['campaign'] = tib['广告活动名称']
search['campaign'] = search['广告活动名称']
search['term'] = search['搜索词'].fillna('').str.strip().str.lower()
range_parts = search['日期范围'].str.extract(r'^(.*?) - (.*?)$')
search['range_start'] = pd.to_datetime(range_parts[0], format='%Y年%m月%d日')
search['range_end'] = pd.to_datetime(range_parts[1], format='%Y年%m月%d日')
search['range_days'] = (search['range_end'] - search['range_start']).dt.days + 1

source_shapes

### 2. Validate grain, uniqueness, coverage, and cross-file totals

In [ ]:
key_tib = ['date', 'hour', '广告活动编号', '广告组编号', '推广的商品编号', '推广的商品 SKU-Advertised product SKU']
key_search = ['广告活动编号', '广告组编号', 'term']

quality = pd.DataFrame({
    'check': [
        'TIB exact duplicate rows', 'TIB duplicate candidate keys',
        'Search exact duplicate rows', 'Search duplicate campaign/ad-group/term keys',
        'TIB campaigns', 'TIB date start', 'TIB date end',
    ],
    'value': [
        int(tib.duplicated().sum()), int(tib.duplicated(key_tib).sum()),
        int(search.duplicated().sum()), int(search.duplicated(key_search).sum()),
        int(tib['campaign'].nunique()), tib['date'].min().date(), tib['date'].max().date(),
    ]
})

tib_totals = tib.groupby('campaign')[MEASURES].sum()
search_totals = search.groupby('campaign')[MEASURES].sum()
reconciliation = (search_totals - tib_totals).rename(columns=lambda c: f'{c}_search_minus_tib')

assert tib.duplicated().sum() == 0
assert tib.duplicated(key_tib).sum() == 0
assert search.duplicated(key_search).sum() == 0
assert np.allclose(search_totals[['点击量','总成本','购买量','销售额']], tib_totals[['点击量','总成本','购买量','销售额']])

display(quality)
display(reconciliation)

搜索词展示量低于 TIB 展示量，但点击、花费、购买和销售完全一致。因此：流量分布统一以 TIB 展示量为准，搜索词表用于点击后质量和查询意图分析。

## Results

### 3. Aggregate campaign performance from raw numerators and denominators

In [ ]:
aliases = {
    'XM2-S75-手动wide广泛': 'S75-wide',
    'XM2-S75-手动sneakers广泛': 'S75-sneakers',
    'XM1-KD5-手动boysshoes广泛': 'KD5-boysshoes',
    'XM1-Y180-手动投BRONAX相关词': 'Y180-BRONAX',
}

def derived(df):
    out = df.copy()
    out['CTR'] = out['点击量'] / out['展示量'].replace(0, np.nan)
    out['CPC'] = out['总成本'] / out['点击量'].replace(0, np.nan)
    out['CVR'] = out['购买量'] / out['点击量'].replace(0, np.nan)
    out['ROAS'] = out['销售额'] / out['总成本'].replace(0, np.nan)
    out['CPA'] = out['总成本'] / out['购买量'].replace(0, np.nan)
    return out

campaign = tib.groupby('campaign', as_index=False)[MEASURES].sum()
campaign['活动'] = campaign['campaign'].map(aliases)
campaign = derived(campaign)
overall = derived(pd.DataFrame([{m: tib[m].sum() for m in MEASURES}]))

display(overall[['展示量','点击量','总成本','购买量','销售额','CTR','CPC','CVR','ROAS','CPA']].round(4))
display(campaign[['活动','展示量','点击量','总成本','购买量','销售额','CPC','CVR','ROAS','CPA']].round(4).sort_values('总成本', ascending=False))

### 4. Compare hourly traffic, efficiency, and coverage

In [ ]:
day_hour = tib.groupby(['date','hour','campaign'], as_index=False)[MEASURES].sum()
hourly = derived(day_hour.groupby('hour', as_index=False)[MEASURES].sum())

display(hourly[['hour','点击量','总成本','购买量','销售额','CVR','ROAS']].round(4))

### 5. Diagnose time truncation from daily spend plateaus and final active hours

In [ ]:
daily = day_hour.groupby(['campaign','date'], as_index=False)[MEASURES].sum()
daily_stats = daily.groupby('campaign', as_index=False).agg(
    mean_daily_spend=('总成本','mean'), sd_daily_spend=('总成本','std'),
    min_daily_spend=('总成本','min'), max_daily_spend=('总成本','max'),
)
daily_stats['spend_cv'] = daily_stats['sd_daily_spend'] / daily_stats['mean_daily_spend']
daily_stats['活动'] = daily_stats['campaign'].map(aliases)

active = day_hour[(day_hour['展示量'] > 0) | (day_hour['点击量'] > 0) | (day_hour['总成本'] > 0)]
last_hour = active.groupby(['campaign','date'], as_index=False)['hour'].max()
last_summary = last_hour.groupby('campaign', as_index=False).agg(
    median_last_active_hour=('hour','median'), min_last_active_hour=('hour','min'), max_last_active_hour=('hour','max')
)
last_summary['活动'] = last_summary['campaign'].map(aliases)

display(daily_stats[['活动','mean_daily_spend','min_daily_spend','max_daily_spend','spend_cv']].round(4))
display(last_summary[['活动','median_last_active_hour','min_last_active_hour','max_last_active_hour']])

### 6. Separate high-confidence bid windows from coverage tests

In [ ]:
campaign_hour = derived(day_hour.groupby(['campaign','hour'], as_index=False)[MEASURES].sum())
stability = day_hour.groupby(['campaign','hour'], as_index=False).agg(
    impression_days=('展示量', lambda s: int((s > 0).sum())),
    click_days=('点击量', lambda s: int((s > 0).sum())),
    order_days=('购买量', lambda s: int((s > 0).sum())),
)
campaign_hour = campaign_hour.merge(stability, on=['campaign','hour'])
campaign_hour['活动'] = campaign_hour['campaign'].map(aliases)

selected = {
    'KD5-boysshoes': [9,10,11,13,18,19,20],
    'Y180-BRONAX': [4,6,7,18,19,20],
    'S75-wide': [8,9,11,12,15,18,19,20],
    'S75-sneakers': [16,18,19,21],
}
rows = []
for alias, hours in selected.items():
    rows.append(campaign_hour[(campaign_hour['活动'] == alias) & campaign_hour['hour'].isin(hours)])
decision_evidence = pd.concat(rows).sort_values(['活动','hour'])
display(decision_evidence[['活动','hour','点击量','总成本','购买量','销售额','ROAS','impression_days','order_days']].round(4))

### 7. Audit search-term quality without dropping short-span terms

In [ ]:
search['zero_order_spend'] = np.where((search['购买量'] == 0) & (search['点击量'] > 0), search['总成本'], 0.0)
search_summary = derived(search.groupby('campaign', as_index=False)[MEASURES].sum())
profiles = search.groupby('campaign', as_index=False).agg(
    unique_terms=('term','nunique'),
    converting_terms=('购买量', lambda s: int((s > 0).sum())),
    zero_order_spend=('zero_order_spend','sum'),
)
search_summary = search_summary.merge(profiles, on='campaign')
search_summary['zero_order_spend_share'] = search_summary['zero_order_spend'] / search_summary['总成本']
search_summary['活动'] = search_summary['campaign'].map(aliases)
display(search_summary[['活动','unique_terms','converting_terms','点击量','总成本','购买量','销售额','ROAS','zero_order_spend_share']].round(4))

top_winners = search[search['购买量'] > 0].sort_values(['销售额','购买量'], ascending=False).groupby('campaign', group_keys=False).head(5).copy()
top_winners['活动'] = top_winners['campaign'].map(aliases)
top_winners['ROAS'] = top_winners['销售额'] / top_winners['总成本']
display(top_winners[['活动','term','点击量','总成本','购买量','销售额','ROAS']].round(4))

## Takeaways

1. **先解决覆盖，再验证晚间竞价。** KD5、Y180、S75-wide 的每日花费和最后活跃小时都呈现明显截断；晚间缺数据不能解释为需求差。
2. **立即小幅加价只限高样本窗口。** 推荐从 +5%～10% 开始，并同时压低同活动的低效窗口，避免加价使预算更早耗尽。
3. **搜索词治理是分时加价的前置护栏。** KD5、S75-wide、S75-sneakers 的无订单搜索词花费占比分别约 81%、60%、65%；在广泛活动上加价会同时放大浪费词。
4. **Y180 最稳定，但集中度最高。** BRONAX 品牌相关查询贡献绝大多数订单；应先迁移高价值词到精准/词组，再对品牌收割层做分时加价。
5. **所有结论均需以盈亏平衡 ROAS 复核。** 当前文件没有商品贡献利润率，报告中的加价窗口是相对效率建议，不代表已确认盈利。

### Validation status

The local bundled Python environment did not include `nbformat`, `nbclient`, or a Jupyter kernel. The notebook was therefore generated as valid Notebook v4 JSON without cached outputs. All code cells were executed sequentially in the same bundled Python process on 2026-08-02 and completed without errors. To populate cell outputs later, run:

```bash
python -m jupyter nbconvert --execute --to notebook --inplace wiki/attachments/XM四广告分段竞价分析-2026-08-02.ipynb
```

This requires a local Jupyter installation with pandas and numpy.